# Демо: плагинные оценщики эпиплексии в `episynth`

Этот блокнот показывает, как пользователь выбирает алгоритм оценки эпиплексии по имени,
как добавить свой, и строит графики по результатам прогона. Для скорости используется
крошечная модель и очень малый бюджет на CPU — достаточно, чтобы продемонстрировать,
что всё считается и рисуется.

Прогон запускается тем же конвейером, что и большие эксперименты в `experiments/episynth`:
конфигурационный CLI `episynth.train` с полем `estimator` в файле конфигурации. Площадь
(model part) считается через `episynth.analysis.run_area` по `window_loss.f32` и
`heldout_ema_1.f32`, как в основном протоколе.

Документация API оценщиков генерируется Sphinx из исходников: после сборки откройте
`doc/_build/api/estimators.html` (из корня репозитория), либо раздел API → Estimators
на [сайте документации](https://intsystems.github.io/Epiplexity/api/estimators.html).
Там показаны сигнатуры, параметры и результаты `PrequentialEstimator.run` и `from_run_dir`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from episynth.analysis import area_curve, run_area
from episynth.estimators import EpiplexityEstimator, available, build, register
from episynth.train import load_run_config

## Доступные оценщики

Встроенные алгоритмы уже зарегистрированы в реестре. `build(name)` выбирает их по имени.

In [ ]:
print("Available estimators:", available())

# Выбор по имени
est = build("prequential")
print("Built:", type(est).__name__, "name =", est.name)

## Конфигурация крошечного прогона

Строим файл конфигурации того же формата, что используют большие серии (`episynth.train`):
разделы `data`, `spec`, `model`, `train` и `estimator`. Модель ограничена снизу числом голов
(`num_heads = d_model // 64`), поэтому берём `d_model = 64`, один слой и одну компоненту.

In [ ]:
config = {
    "data": {"generator": "waveform"},
    "spec": {
        "series_length": 256, "context_max": 128, "horizon": 32,
        "patch_size": 32, "context_lengths": [128], "dequantize_width": 1e-3,
    },
    "model": {"d_model": 64, "num_layers": 1, "max_seq_len": 32,
              "patch_sizes": [32], "components": ["student_t"]},
    "train": {
        "n_windows": 64, "batch_size": 4, "device": "cpu", "warmup_steps": 0,
        "precision": "fp32",
        "checkpoint_fractions": [1.0], "heldout_windows": 32, "heldout_first_seed": 100000,
        "first_seed": 0, "log_every": 8, "stream_workers": 0,
    },
    "estimator": "prequential",
}

out_dir = Path("runs/demo_prequential")
cfg_path = Path("runs/demo_config.json")
cfg_path.parent.mkdir(parents=True, exist_ok=True)
cfg_path.write_text(json.dumps(config, indent=1))

cfg = load_run_config(cfg_path)
print("estimator:", cfg.estimator)
print("n_windows:", cfg.train.n_windows)

## Прогон выбранного оценщика

Тот же код, что в `python -m episynth.train` (см. `src/episynth/train.py`): строит генератор
и модель, вызывает `build_estimator(estimator).run(...)` и пишет `estimator_result.json`.
Ниже выполняем те же шаги программно; CLI дополнительно сохраняет `estimator_result.json`.

In [ ]:
from episynth.train import build_generator, build_model, check_seed_ranges

generator = build_generator(cfg.data, cfg.spec.series_length)
check_seed_ranges(generator, cfg.train)
model = build_model(cfg.model, cfg.train.init_seed)
print("model parameters:", sum(p.numel() for p in model.parameters()))

result = build(cfg.estimator).run(model, generator, cfg.spec, cfg.train, out_dir)
print(json.dumps(result, indent=2))

## Графики по результатам прогона

Кривая обучения (площадь над финальным лоссом закрашена) и разложение на model/data part
в битах. Площадь и финальный лосс берём через `analysis.run_area`/`area_curve`, как в
больших сериях.

In [ ]:
import numpy as np

loss = np.memmap(out_dir / "window_loss.f32", dtype=np.float32, mode="r",
                 shape=(cfg.train.n_windows,))
ns, curve = area_curve(out_dir, variant="ema", points=64)
area = run_area(out_dir, variant="ema")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(loss, lw=1)
ax[0].axhline(area.l_fin, color="r", ls="--", label=f"final loss ({area.l_fin:.3f})")
ax[0].fill_between(range(len(loss)), area.l_fin, loss, where=loss > area.l_fin, color="C0", alpha=0.3)
ax[0].set_title("Per-window loss (nats)")
ax[0].set_xlabel("window index")
ax[0].legend()

ax[1].bar(["model part", "data part"],
          [result["model_part_bits"], result["data_part_bits"]],
          color=["C0", "C2"])
ax[1].set_title("Epiplexity estimate (bits)")
ax[1].set_ylabel("bits")
plt.tight_layout()
plt.show()

## Добавление собственного оценщика одним модулем

Нужно реализовать контракт `EpiplexityEstimator` (метод `run`) и зарегистрировать его
декоратором `@register`. Остальные модули не меняются — выбранный оценщик подставляется в
поле `estimator` конфигурации `episynth.train`.

In [ ]:
@register("demo_constant")
class ConstantEstimator(EpiplexityEstimator):
    """Тривиальный пример: возвращает нулевую оценку без обучения."""
    name = "demo_constant"

    def run(self, model, generator, spec, config, out_dir):
        return {"model_part_bits": 0.0, "data_part_bits": 0.0}


print("Available after registration:", available())
print("Built:", build("demo_constant").run(None, None, None, None, "unused"))